## Example 1: parse a variable declaration into a syntax tree

In [ ]:
import pyjsparser

# parse() returns the syntax tree as plain dicts and lists, in the ESTree layout that JavaScript tools use.
tree = pyjsparser.parse("var x = 1 + 2;")
assert tree["type"] == "Program", tree["type"]
declaration = tree["body"][0]
assert declaration["type"] == "VariableDeclaration" and declaration["kind"] == "var", declaration
declarator = declaration["declarations"][0]
assert declarator["id"] == {"type": "Identifier", "name": "x"}, declarator["id"]
total = declarator["init"]
assert total["type"] == "BinaryExpression" and total["operator"] == "+", total
# A numeric literal keeps its source text in "raw"; its value is a JavaScript number, which arrives as a float.
assert (total["left"]["value"], total["left"]["raw"], total["right"]["value"], total["right"]["raw"]) == (1, "1", 2, "2"), total
print("var x = 1 + 2 ->", tree["type"], ">", declaration["type"], ">", total["type"], repr(total["operator"]))

## Example 2: a function declaration and a call

In [ ]:
import pyjsparser

tree = pyjsparser.parse("function add(a, b) { return a + b; }\nadd(2, 3);")
# The parser adds a ';' after the source before it starts, so a tree can end with an extra EmptyStatement.
statements = [node for node in tree["body"] if node["type"] != "EmptyStatement"]
assert [node["type"] for node in statements] == ["FunctionDeclaration", "ExpressionStatement"], statements
function, call = statements
assert function["id"]["name"] == "add" and [param["name"] for param in function["params"]] == ["a", "b"], function
returned = function["body"]["body"][0]
assert returned["type"] == "ReturnStatement" and returned["argument"]["type"] == "BinaryExpression", returned
assert [returned["argument"]["left"]["name"], returned["argument"]["right"]["name"]] == ["a", "b"], returned
expression = call["expression"]
assert expression["type"] == "CallExpression" and expression["callee"]["name"] == "add", expression
assert [argument["value"] for argument in expression["arguments"]] == [2, 3], expression["arguments"]
print("function", function["id"]["name"], "with parameters", [param["name"] for param in function["params"]],
      "is called with", [argument["raw"] for argument in expression["arguments"]])

## Example 3: syntax errors raise JsSyntaxError

In [ ]:
import pyjsparser

# Invalid code raises pyjsparser.JsSyntaxError, and the message starts with the line of the error.
for source, line in (("var = 1;", 1), ("var ok = 1;\nvar = 2;", 2), ("x = {;", 1)):
    try:
        pyjsparser.parse(source)
    except pyjsparser.JsSyntaxError as error:
        assert str(error).startswith(f"Line {line}:"), (source, str(error))
        print(f"{source!r:28} -> {error}")
    else:
        raise AssertionError(f"{source!r} was parsed without an error")
# pyjsparser implements ECMAScript 5.1, so newer syntax such as an arrow function is refused the same way.
try:
    pyjsparser.parse("var double = (n) => n * 2;")
except pyjsparser.JsSyntaxError as error:
    assert "ArrowFunction" in str(error), str(error)
    print(f"{'arrow function':28} -> {error}")
else:
    raise AssertionError("an arrow function was parsed")

## Example 4: objects, arrays, control flow and string escapes

In [ ]:
import pyjsparser

source = ('var config = {name: "Ada", tags: ["a", "b"], size: 0x10};\n'
          "if (config.size > 3) { config.size -= 1; } else { config.size = 0; }\n"
          'for (var i = 0; i < 3; i++) { config.tags.push("t" + i); }\n'
          'var text = "line\\n\\u0041";\n')
statements = [node for node in pyjsparser.parse(source)["body"] if node["type"] != "EmptyStatement"]
assert [node["type"] for node in statements] == ["VariableDeclaration", "IfStatement", "ForStatement", "VariableDeclaration"], statements
config = statements[0]["declarations"][0]["init"]
assert config["type"] == "ObjectExpression" and [prop["key"]["name"] for prop in config["properties"]] == ["name", "tags", "size"]
name, tags, size = (prop["value"] for prop in config["properties"])
assert name["value"] == "Ada" and [element["value"] for element in tags["elements"]] == ["a", "b"], (name, tags)
assert size["value"] == 16 and size["raw"] == "0x10", size                        # hexadecimal literal, value converted
branch = statements[1]
assert branch["test"]["operator"] == ">" and branch["test"]["left"]["type"] == "MemberExpression", branch["test"]
assert branch["consequent"]["body"][0]["expression"]["operator"] == "-=" and branch["alternate"] is not None
loop = statements[2]
assert loop["update"] == {"type": "UpdateExpression", "operator": "++", "argument": {"type": "Identifier", "name": "i"}, "prefix": False}, loop["update"]
assert loop["body"]["body"][0]["expression"]["callee"]["property"]["name"] == "push"
text = statements[3]["declarations"][0]["init"]
assert text["value"] == "line\nA" and text["raw"] == '"line\\n\\u0041"', text     # escapes decoded in value, kept in raw
print("statements:", [node["type"] for node in statements], "| size", size["raw"], "->", size["value"], "| text", text["raw"], "->", repr(text["value"]))

## Example 5: walk the tree, and reuse one parser object

In [ ]:
import collections
import pyjsparser

def walk(node):
    """Yield every node of a pyjsparser tree: each dict that has a "type", at any depth."""
    if isinstance(node, dict):
        if "type" in node:
            yield node
        for value in node.values():
            yield from walk(value)
    elif isinstance(node, list):
        for item in node:
            yield from walk(item)

source = """function greet(person) {
  var message = "Hello, " + person.name;
  return message.toUpperCase();
}
var people = [{name: "Ada"}, {name: "Linus"}];
for (var i = 0; i < people.length; i++) { greet(people[i]); }
"""
parser = pyjsparser.PyJsParser()                                   # one parser object can parse many sources
tree = parser.parse(source)
counts = collections.Counter(node["type"] for node in walk(tree))
names = sorted({node["name"] for node in walk(tree) if node["type"] == "Identifier"})
strings = sorted(node["value"] for node in walk(tree) if node["type"] == "Literal" and isinstance(node["value"], str))
assert counts["FunctionDeclaration"] == 1 and counts["ForStatement"] == 1 and counts["CallExpression"] == 2, counts
assert names == ["greet", "i", "length", "message", "name", "people", "person", "toUpperCase"], names
assert strings == ["Ada", "Hello, ", "Linus"], strings
again = parser.parse("var answer = 42;")
assert again["body"][0]["declarations"][0]["init"]["value"] == 42, again
print(f"{sum(counts.values())} nodes of {len(counts)} kinds; identifiers {names}; string literals {strings}")